# G0 Compare — 여러 sweep / strategy 간 side-by-side 비교

**용도**: seq sweep 여러 라운드 (g0_00 vs g0_03_thp 등) 또는 strategy 3종 비교 + **gpu_only baseline 대조**.

**Usage**: `ROOTS` 리스트에 비교할 각 sweep dir 절대경로 나열 후 전체 실행.
각 dir 안의 `gpu_only_baseline/gpu_only.json` 이 있으면 자동으로 baseline 으로 함께 표시.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ─── 비교할 sweep dir 들 ─────────────────────────────────────────────
ROOTS = [
    Path('/vllm_hybrid/measurement_results/H100x8/g0_02_strat_capacity'),
    Path('/vllm_hybrid/measurement_results/H100x8/g0_02_strat_length_aware'),
    Path('/vllm_hybrid/measurement_results/H100x8/g0_02_strat_throughput_adaptive'),
]
# ──────────────────────────────────────────────────────────────────────

for r in ROOTS:
    seqs = sorted(int(m.group(1)) for d in r.iterdir() if d.is_dir()
                  for m in [re.match(r'seqs(\d+)$', d.name)] if m)
    has_baseline = (r / 'gpu_only_baseline' / 'gpu_only.json').exists()
    print(f'{r.name}: seqs={seqs}  baseline={"✅" if has_baseline else "❌"}')

## 1. Bench 결과 수집 (hybrid + gpu_only baseline)

In [ ]:
def load_bench_hybrid(root):
    rows = []
    for d in sorted(root.iterdir()):
        m = re.match(r'seqs(\d+)$', d.name)
        if not m: continue
        s = int(m.group(1))
        for mode in ('hybrid', 'gpu_only'):
            jp = d / f'{mode}.json'
            if jp.exists():
                b = json.loads(jp.read_text())
                rows.append({
                    'run': root.name, 'seqs': s, 'mode': mode,
                    'wall_s': b.get('duration'),
                    'out_tp': b.get('output_throughput'),
                    'tpot_mean': b.get('mean_tpot_ms'),
                    'tpot_p99': b.get('p99_tpot_ms'),
                    'ttft_p99': b.get('p99_ttft_ms'),
                })
                break
    return rows

def load_gpu_only_baseline(root):
    jp = root / 'gpu_only_baseline' / 'gpu_only.json'
    if not jp.exists(): return None
    b = json.loads(jp.read_text())
    return {
        'run': root.name, 'seqs': 0, 'mode': 'gpu_only_baseline',
        'wall_s': b.get('duration'),
        'out_tp': b.get('output_throughput'),
        'tpot_mean': b.get('mean_tpot_ms'),
        'tpot_p99': b.get('p99_tpot_ms'),
        'ttft_p99': b.get('p99_ttft_ms'),
    }

all_rows = []
baseline_rows = []
for r in ROOTS:
    all_rows.extend(load_bench_hybrid(r))
    b = load_gpu_only_baseline(r)
    if b is not None:
        baseline_rows.append(b)
bdf = pd.DataFrame(all_rows)      # hybrid seq sweep
gbdf = pd.DataFrame(baseline_rows)  # gpu_only baselines
print('hybrid rows:', len(bdf), 'gpu_only baselines:', len(gbdf))
print('\n=== gpu_only baselines ===')
display(gbdf) if not gbdf.empty else print('(none)')

## 2. Wall / throughput 비교 (seqs 별) + gpu_only 기준선

각 run 의 `gpu_only_baseline` 은 **같은 색 가로 점선** 으로 overlay.

In [ ]:
if not bdf.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    metrics = [('wall_s', 'Wall (s) — (lower is better)', 'linear'),
               ('out_tp', 'Output TP (tok/s) — (higher is better)', 'log'),
               ('tpot_p99', 'TPOT p99 (ms) — (lower is better)', 'log')]
    runs = bdf['run'].unique()
    cmap = {run: plt.cm.tab10(i) for i, run in enumerate(runs)}
    for ax, (col, title, yscale) in zip(axes, metrics):
        for run in runs:
            d = bdf[bdf['run'] == run].sort_values('seqs')
            ax.plot(d['seqs'], d[col], 'o-', label=run.replace('g0_', ''),
                    markersize=8, linewidth=1.5, color=cmap[run])
            # gpu_only baseline 가로 점선
            if not gbdf.empty:
                gb = gbdf[gbdf['run'] == run]
                if not gb.empty:
                    ax.axhline(y=gb[col].iloc[0], color=cmap[run],
                               linestyle='--', alpha=0.6, linewidth=1,
                               label=f'{run.replace("g0_","")} gpu_only')
        ax.set_xscale('log', base=2)
        ax.set_yscale(yscale)
        if bdf['seqs'].nunique() > 0:
            ax.set_xticks(sorted(bdf['seqs'].unique()))
            ax.set_xticklabels([str(int(s)) for s in sorted(bdf['seqs'].unique())])
        ax.set_xlabel('seqs'); ax.set_title(title)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/tmp/g0_compare_bench.png', dpi=130, bbox_inches='tight')
    plt.show()

## 3. Router stats (CPU 처리 비율) — hybrid 만

`hybrid_server_run.log` 의 마지막 `[HYBRID-ROUTER-STATS]` 라인 파싱.

In [ ]:
ROUTER_RE = re.compile(
    r'finished=(\d+) GPU=([\d.]+) tok/s \((\d+) reqs\), '
    r'CPU=([\d.]+) tok/s \((\d+) reqs\), cpu_ratio=([\d.]+)%')

router_rows = []
for r in ROOTS:
    for d in sorted(r.iterdir()):
        m = re.match(r'seqs(\d+)$', d.name)
        if not m: continue
        s = int(m.group(1))
        log = d / 'hybrid_server_run.log'
        if not log.exists(): continue
        last = None
        for line in log.read_text(errors='replace').splitlines():
            mm = ROUTER_RE.search(line)
            if mm: last = mm.groups()
        if last:
            router_rows.append({
                'run': r.name, 'seqs': s, 'finished': int(last[0]),
                'gpu_tps': float(last[1]), 'gpu_reqs': int(last[2]),
                'cpu_tps': float(last[3]), 'cpu_reqs': int(last[4]),
                'cpu_ratio_%': float(last[5]),
            })
rdf = pd.DataFrame(router_rows)
rdf

## 4. CPU / GPU utilization 수집 (hybrid + baseline)

In [ ]:
def _n_physical_from_sys_info(sys_info_path):
    if not sys_info_path.exists():
        return None
    si = json.loads(sys_info_path.read_text())
    c = si.get('cpu', {})
    return c.get('sockets', 1) * c.get('cores_per_socket', 1)

def _phys_cpu_cols(df, n_phys):
    if n_phys is None:
        return [c for c in df.columns
                if c.startswith('cpu') and c.endswith('_util_pct')
                and c != 'cpu_avg_util_pct']
    return [f'cpu{i}_util_pct' for i in range(n_phys)
            if f'cpu{i}_util_pct' in df.columns]

def load_util_row(run, seqs, mode, cpu_p, gpu_p):
    if not (cpu_p.exists() and gpu_p.exists()): return None
    cdf = pd.read_csv(cpu_p); gdf = pd.read_csv(gpu_p)
    # system_info 는 cpu_p 의 부모 디렉토리에서 로드
    n_phys = _n_physical_from_sys_info(cpu_p.parent / 'system_info.json')
    pc = _phys_cpu_cols(cdf, n_phys)
    cpu_mean_series = cdf[pc].mean(axis=1) if pc else cdf['cpu_avg_util_pct']
    return {
        'run': run, 'seqs': seqs, 'mode': mode,
        'cpu_mean_%': cpu_mean_series.mean(),
        'cpu_max_%': cpu_mean_series.max(),
        'n_physical': n_phys,
        'gpu_mean_%': gdf['gpu_avg_util_pct'].mean(),
        'gpu_max_%': gdf['gpu_avg_util_pct'].max(),
        'gpu_power_mean_W': gdf['gpu_avg_power_w'].mean(),
    }

util_rows = []
for r in ROOTS:
    # hybrid sweep
    for d in sorted(r.iterdir()):
        m = re.match(r'seqs(\d+)$', d.name)
        if not m: continue
        row = load_util_row(r.name, int(m.group(1)), 'hybrid',
                             d / 'hybrid_monitor_cpu.csv',
                             d / 'hybrid_monitor_gpu.csv')
        if row: util_rows.append(row)
    # gpu_only baseline
    bd = r / 'gpu_only_baseline'
    if bd.exists():
        row = load_util_row(r.name, 0, 'gpu_only_baseline',
                             bd / 'gpu_only_monitor_cpu.csv',
                             bd / 'gpu_only_monitor_gpu.csv')
        if row: util_rows.append(row)
udf = pd.DataFrame(util_rows).round(2)
udf

### 4.1 GPU avg util 시계열 (hybrid + baseline overlay)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for r in ROOTS:
    for d in sorted(r.iterdir()):
        m = re.match(r'seqs(\d+)$', d.name)
        if not m: continue
        gpu_p = d / 'hybrid_monitor_gpu.csv'
        if not gpu_p.exists(): continue
        g = pd.read_csv(gpu_p)
        ax.plot(g['elapsed_s'], g['gpu_avg_util_pct'],
                label=f"{r.name.replace('g0_','')}/s{m.group(1)}",
                alpha=0.7, linewidth=1)
    # baseline (굵은 점선)
    bd = r / 'gpu_only_baseline' / 'gpu_only_monitor_gpu.csv'
    if bd.exists():
        g = pd.read_csv(bd)
        ax.plot(g['elapsed_s'], g['gpu_avg_util_pct'],
                label=f"{r.name.replace('g0_','')}/baseline",
                linestyle='--', linewidth=2, alpha=0.9)
ax.set_xlabel('elapsed (s)'); ax.set_ylabel('GPU avg util (%)')
ax.set_ylim(0, 105); ax.set_title('GPU avg utilization — hybrid × seqs + gpu_only baseline (dashed)')
ax.legend(fontsize=7, ncol=3, loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/g0_compare_gpu_util.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. 요약 비교 표 (hybrid + baseline)

In [ ]:
# hybrid: bench + router + util 병합
summary = bdf.merge(rdf[['run','seqs','cpu_ratio_%','cpu_reqs']], on=['run','seqs'], how='left')
summary = summary.merge(udf[['run','seqs','cpu_mean_%','gpu_mean_%','gpu_power_mean_W']],
                         on=['run','seqs'], how='left')
# baseline: bench + util 만 (router 없음)
if not gbdf.empty:
    gsum = gbdf.merge(udf[udf['mode']=='gpu_only_baseline'][
        ['run','seqs','cpu_mean_%','gpu_mean_%','gpu_power_mean_W']],
        on=['run','seqs'], how='left')
    gsum['cpu_ratio_%'] = 0.0; gsum['cpu_reqs'] = 0
    summary = pd.concat([summary, gsum], ignore_index=True)
cols = ['run','mode','seqs','wall_s','out_tp','tpot_p99',
        'cpu_ratio_%','cpu_reqs','cpu_mean_%','gpu_mean_%','gpu_power_mean_W']
summary = summary[cols].sort_values(['run','mode','seqs']).round(2)
summary

## 6. Bench metric bar chart + baseline 수평선

wall / out_tp / tpot_p99 bar — gpu_only baseline 은 가로 점선.

In [ ]:
if not bdf.empty:
    metrics = [('wall_s', 'Wall (s) ↓'),
               ('out_tp', 'Output TP (tok/s) ↑'),
               ('tpot_p99', 'TPOT p99 (ms) ↓')]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    runs = bdf['run'].unique()
    cmap = {run: plt.cm.tab10(i) for i, run in enumerate(runs)}
    for ax, (col, title) in zip(axes, metrics):
        pv = bdf.pivot_table(index='seqs', columns='run', values=col)
        pv.plot(kind='bar', ax=ax, width=0.8,
                color=[cmap[c] for c in pv.columns])
        # baseline 가로선
        if not gbdf.empty:
            for run in runs:
                gb = gbdf[gbdf['run'] == run]
                if not gb.empty:
                    ax.axhline(y=gb[col].iloc[0], color=cmap[run],
                               linestyle='--', alpha=0.6, linewidth=1.2)
        ax.set_title(title); ax.set_xlabel('seqs')
        ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.savefig('/tmp/g0_compare_bars.png', dpi=130, bbox_inches='tight')
    plt.show()

## 7. Router: CPU 기여도 (run × seqs) — hybrid 만

In [ ]:
if not rdf.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for col, ax, title in [('cpu_ratio_%', axes[0], 'CPU share (%)'),
                            ('cpu_reqs', axes[1], 'CPU completed reqs')]:
        pv = rdf.pivot_table(index='seqs', columns='run', values=col)
        pv.plot(kind='bar', ax=ax, width=0.8)
        ax.set_title(title); ax.set_xlabel('seqs')
        ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)
        ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.savefig('/tmp/g0_compare_router.png', dpi=130, bbox_inches='tight')
    plt.show()

## 8. Per-CPU heatmap — run 별 + baseline tile

각 run 의 대표 seqs (최대) + gpu_only baseline 의 per-core util 시계열.

In [ ]:
tiles = []
for r in ROOTS:
    seqs_dirs = sorted([d for d in r.iterdir() if re.match(r'seqs(\d+)$', d.name)],
                        key=lambda d: int(re.match(r'seqs(\d+)$', d.name).group(1)))
    if seqs_dirs:
        d = seqs_dirs[-1]
        cpu_p = d / 'hybrid_monitor_cpu.csv'
        if cpu_p.exists():
            tiles.append((f'{r.name} / {d.name}', pd.read_csv(cpu_p),
                          _n_physical_from_sys_info(d / 'system_info.json')))
    bd = r / 'gpu_only_baseline' / 'gpu_only_monitor_cpu.csv'
    if bd.exists():
        tiles.append((f'{r.name} / baseline', pd.read_csv(bd),
                      _n_physical_from_sys_info(bd.parent / 'system_info.json')))

if tiles:
    n = len(tiles)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3*n))
    if n == 1: axes = [axes]
    for ax, (title, df, n_phys) in zip(axes, tiles):
        cpu_cols = _phys_cpu_cols(df, n_phys)
        mat = df[cpu_cols].T.values
        im = ax.imshow(mat, aspect='auto', cmap='hot', interpolation='nearest',
                       vmin=0, vmax=100,
                       extent=[df['elapsed_s'].iloc[0], df['elapsed_s'].iloc[-1],
                               len(cpu_cols), 0])
        ax.set_xlabel('elapsed (s)'); ax.set_ylabel('CPU core #')
        ax.set_title(f'{title}  (n_cpus={len(cpu_cols)})')
        plt.colorbar(im, ax=ax, label='util %')
    plt.tight_layout()
    plt.savefig('/tmp/g0_compare_cpu_heatmap.png', dpi=130, bbox_inches='tight')
    plt.show()

## 9. GPU per-device utilization (8 GPUs)

각 run 의 대표 seqs + baseline 에서 per-GPU util.

In [ ]:
for r in ROOTS:
    seqs_dirs = sorted([d for d in r.iterdir() if re.match(r'seqs(\d+)$', d.name)],
                        key=lambda d: int(re.match(r'seqs(\d+)$', d.name).group(1)))
    plots = []
    if seqs_dirs:
        d = seqs_dirs[-1]
        gp = d / 'hybrid_monitor_gpu.csv'
        if gp.exists(): plots.append((d.name, pd.read_csv(gp)))
    bd = r / 'gpu_only_baseline' / 'gpu_only_monitor_gpu.csv'
    if bd.exists(): plots.append(('baseline', pd.read_csv(bd)))
    if not plots: continue
    fig, axes = plt.subplots(1, len(plots), figsize=(7*len(plots), 4), sharey=True)
    if len(plots) == 1: axes = [axes]
    for ax, (label, g) in zip(axes, plots):
        gpu_cols = [c for c in g.columns if re.match(r'gpu\d+_util_pct$', c)]
        for c in gpu_cols:
            ax.plot(g['elapsed_s'], g[c], label=c.split('_')[0], alpha=0.8, linewidth=1)
        ax.set_xlabel('elapsed (s)'); ax.set_ylabel('util %'); ax.set_ylim(0, 105)
        ax.set_title(f'{r.name} / {label}')
        ax.legend(fontsize=7, ncol=len(gpu_cols), loc='upper right')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Pareto — wall vs CPU 기여도 + gpu_only 기준점

gpu_only baseline = X축 0%, 가장 낮은 wall (★ 별표로 표시).

In [ ]:
merged = bdf.merge(rdf[['run','seqs','cpu_ratio_%','cpu_reqs']], on=['run','seqs'], how='inner')
if not merged.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    runs = bdf['run'].unique()
    cmap = {run: plt.cm.tab10(i) for i, run in enumerate(runs)}
    # hybrid points
    for run in runs:
        rows = merged[merged['run']==run]
        ax.scatter(rows['cpu_ratio_%'], rows['wall_s'],
                   s=80, color=cmap[run], edgecolor='black', linewidth=0.5,
                   label=run.replace('g0_',''))
        for _, row in rows.iterrows():
            ax.annotate(f"s={int(row['seqs'])}", (row['cpu_ratio_%'], row['wall_s']),
                        fontsize=7, xytext=(5, 3), textcoords='offset points')
    # gpu_only baseline (★)
    if not gbdf.empty:
        for _, row in gbdf.iterrows():
            ax.scatter(0, row['wall_s'], s=200, marker='*',
                       color=cmap.get(row['run'], 'black'),
                       edgecolor='black', linewidth=1,
                       label=f"{row['run'].replace('g0_','')} baseline")
            ax.annotate(f"base {row['wall_s']:.1f}s", (0, row['wall_s']),
                        fontsize=8, xytext=(5, -10), textcoords='offset points')
    ax.set_xlabel('CPU share (%)'); ax.set_ylabel('wall (s) — (lower is better)')
    ax.set_title('Pareto: CPU contribution vs wall (star = gpu_only baseline)')
    ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/tmp/g0_compare_pareto.png', dpi=130, bbox_inches='tight')
    plt.show()